In [18]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,Activity ID,Activity Date,Distance,Elapsed Time,Moving Time,Starting Latitude,Starting Longitude,Finish Latitude,Finish Longitude,Label
0,15561313085,"Aug 23, 2025, 4:55:36 PM",11.19,4332,2900.0,44.386880,38.845067,44.386660,38.845216,PURE_LEISURE
1,16321410676,"Nov 1, 2025, 10:58:07 AM",10.51,9262,2724.0,44.387036,38.843094,44.403151,38.781384,PURE_LEISURE
2,12867426384,"Nov 10, 2024, 10:18:40 AM",19.10,8214,5130.0,44.386809,38.845026,44.386922,38.845124,PURE_LEISURE
3,11903829345,"Jul 16, 2024, 4:02:30 PM",15.31,5822,4259.0,44.387054,38.811028,44.386923,38.845091,LEISURELY_COMMUTE
4,14323523606,"Apr 29, 2025, 3:48:33 PM",4.46,1383,1194.0,44.387410,38.811053,44.386965,38.845283,COMMUTE


In [19]:
df_train["Month"] = df_train["Activity Date"].map(lambda x : str(x).split(' ')[0])
df_train["Speed"] = df_train.Distance/(df_train["Moving Time"] / 3600)
df_test["Speed"] = df_test.Distance/(df_test["Moving Time"] / 3600)

subtask1 = pd.DataFrame(df_train.groupby('Month')["Speed"].mean())

df_1 = pd.DataFrame({
    "subtaskID": 1,
    "Answer1": subtask1.index,
    "Answer2": subtask1.Speed
})

df_1 = df_1.reset_index().drop(columns='Month')
df_1

,subtaskID,Answer1,Answer2
0,1,Apr,12.877157
1,1,Aug,13.976657
2,1,Dec,13.133613
3,1,Feb,11.448271
4,1,Jan,14.453876
5,1,Jul,13.627172
6,1,Jun,13.283836
7,1,Mar,12.993495
8,1,May,12.645787
9,1,Nov,13.880247


In [20]:
df_train.head()

,Activity ID,Activity Date,Distance,Elapsed Time,Moving Time,Starting Latitude,Starting Longitude,Finish Latitude,Finish Longitude,Label,Month,Speed
0,15561313085,"Aug 23, 2025, 4:55:36 PM",11.19,4332,2900.0,44.386880,38.845067,44.386660,38.845216,PURE_LEISURE,Aug,13.891034
1,16321410676,"Nov 1, 2025, 10:58:07 AM",10.51,9262,2724.0,44.387036,38.843094,44.403151,38.781384,PURE_LEISURE,Nov,13.889868
2,12867426384,"Nov 10, 2024, 10:18:40 AM",19.10,8214,5130.0,44.386809,38.845026,44.386922,38.845124,PURE_LEISURE,Nov,13.403509
3,11903829345,"Jul 16, 2024, 4:02:30 PM",15.31,5822,4259.0,44.387054,38.811028,44.386923,38.845091,LEISURELY_COMMUTE,Jul,12.941066
4,14323523606,"Apr 29, 2025, 3:48:33 PM",4.46,1383,1194.0,44.387410,38.811053,44.386965,38.845283,COMMUTE,Apr,13.447236


In [21]:
df_train.columns

Index(['Activity ID', 'Activity Date', 'Distance', 'Elapsed Time',
       'Moving Time', 'Starting Latitude', 'Starting Longitude',
       'Finish Latitude', 'Finish Longitude', 'Label', 'Month', 'Speed'],
      dtype='object')

In [22]:
features = ['Distance', 'Elapsed Time', 'Moving Time', 'Starting Latitude', 
            'Starting Longitude','Finish Latitude', 'Finish Longitude', 'Speed']
target = 'Label'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

X.head()

,Distance,Elapsed Time,Moving Time,Starting Latitude,Starting Longitude,Finish Latitude,Finish Longitude,Speed
0,11.19,4332,2900.0,44.386880,38.845067,44.386660,38.845216,13.891034
1,10.51,9262,2724.0,44.387036,38.843094,44.403151,38.781384,13.889868
2,19.10,8214,5130.0,44.386809,38.845026,44.386922,38.845124,13.403509
3,15.31,5822,4259.0,44.387054,38.811028,44.386923,38.845091,12.941066
4,4.46,1383,1194.0,44.387410,38.811053,44.386965,38.845283,13.447236


In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

model = RandomForestClassifier()

cross_val_score(model, X, y, cv=3, scoring='accuracy')

array([0.8961039 , 0.96052632, 0.85526316])

In [31]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [32]:
df_preds = pd.DataFrame({
    "subtaskID": 2,
    "Answer1": df_test['Activity ID'],
    "Answer2": y_preds
})

df_preds.head()

,subtaskID,Answer1,Answer2
0,2,14319106971,COMMUTE
1,2,12589508252,PURE_LEISURE
2,2,11908121839,COMMUTE
3,2,16687631914,PURE_LEISURE
4,2,12212251514,PURE_LEISURE


In [33]:
submission = pd.concat([df_1, df_preds])
submission

,subtaskID,Answer1,Answer2
0,1,Apr,12.877157
1,1,Aug,13.976657
2,1,Dec,13.133613
3,1,Feb,11.448271
4,1,Jan,14.453876
...,...,...,...
95,2,14537059131,COMMUTE
96,2,13500405361,PURE_LEISURE
97,2,15589048727,COMMUTE
98,2,15174665209,PURE_LEISURE


In [34]:
submission.to_csv("submission.csv", index=False)